In [1]:
from gen_dataset import simulate_imgs

GEN_DATASET: bool = False

dirpath: str = '/home/edoardo/Desktop/ImgsMockDatasetDMs'
# dirpath: str = '/mnt/d/DL_test_dataset'

if GEN_DATASET:
    num_imgs: int = -1
    polygonVertices: int = 6

    simulate_imgs(
        num_imgs=num_imgs,
        polygonVertices=polygonVertices,
        save_to_dir=dirpath,
        start_num=0,
    )
else:
    print('Dataset already generated!')

Dataset already generated!


In [ ]:
from typing import Any, Callable
from pathlib import Path
from PIL import Image
import random

from tqdm import tqdm
import torch
from torch.types import Tensor
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms

In [3]:
def get_filespaths(dirpath: str | Path, data_frmt: str = 'png', shuffle: bool = True) -> list[str]:
    """Groups all the data file-paths inside the given directory."""
    dirpath_ = Path(dirpath)
    paths_list: list[str] = sorted([str(path) for path in dirpath_.glob(f'*.{data_frmt}')])
    if shuffle:
        for _ in range(10): random.shuffle(paths_list)
    return paths_list


def get_dataset(
    data_path: str | Path,
    batch_size: int,
    configurator: Callable[[list[str]], Dataset],
    valid_size: float | None = None,
    data_frmt: str = 'png',
    shuffle: bool = True,
) -> tuple[DataLoader, DataLoader | None]:
    """Dataset generation with given data pre-processing."""
    if valid_size and not (0 <= valid_size < 1):
        raise ValueError(f"Invalid 'valid_size' value {valid_size}, must be in [0, 1).")

    # load data files paths
    paths_list = get_filespaths(data_path, data_frmt, shuffle)
    # config dataset
    print('Baking the dataset...')
    dataset = configurator(paths_list)
    # split dataset in train/validation (if given `valid_size`)
    if valid_size is not None:
        v = int(valid_size * len(dataset))
        validation = Subset(dataset, torch.arange(v))
        training = Subset(dataset, torch.arange(v, len(dataset)))
        valid_dataset = DataLoader(validation, batch_size, shuffle=False)
        train_dataset = DataLoader(training, batch_size, shuffle=True)
    else:
        valid_dataset = None
        train_dataset = DataLoader(dataset, batch_size, shuffle=True)
    print('Dataset ready-to-go!')

    return train_dataset, valid_dataset


def save_dataset(
    dataset: DataLoader,
    save_to: str | Path,
    overwrite: bool = False,
) -> None:
    """Saves given dataset to '.pt' file."""
    if Path(save_to).is_file() and not overwrite:
        print("Dataset already saved!")
        return None
    print("Saving dataset...")
    torch.save(dataset, save_to)
    print("Dataset saved!")
    return None


def load_dataset(filepath: str | Path) -> DataLoader:
    """Load given dataset from '.pt' file."""
    print("Loading dataset...")
    dataset: DataLoader = torch.load(filepath, weights_only=False)
    print("Dataset loaded!")
    return dataset

In [4]:
def center_tensor(tensor: Tensor, eps: float = 1e-8) -> Tensor:
    """Centers given tensor of shape `[C, H, W]` per-channel by subtracting the mean and dividing by the std."""
    dims = (1, 2)
    tensor_ = tensor.to(torch.float)
    mu = tensor_.mean(dims, keepdim=True)
    std = tensor_.std(dims, keepdim=True)
    centered = (tensor_ - mu) / (std + eps)
    return centered

def normalise(tensor: Tensor, norm_range: str = 'unilateral') -> Tensor:
    """Normalises given tensor in the range [0, 1] or [-1, 1]."""
    if norm_range not in ['unilateral', 'bilateral']:
        raise ValueError(f"Invalid 'norm_range' {norm_range}.")
    normalised = (tensor - tensor.min()) / (tensor.max() - tensor.min())
    if norm_range == 'bilateral':
        normalised = 2 * normalised - 1.0
    return normalised

In [5]:
class ImageDataset(Dataset):
    """
    Baseline AutoEncoder dataset configurator.
    """
    def __init__(self, files_list: list[str], preprocess: list[Callable] | None) -> None:
        transform: Callable = (
            preprocess if preprocess is not None else lambda x: transforms.PILToTensor(x)
        )
        self.data = [
            transform(Image.open(img).convert('RGB')) for img in tqdm(files_list)
        ]
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index) -> tuple[Tensor, Tensor]:
        sample = self.data[index]
        return sample, sample

In [6]:
BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
BATCH_SIZE: int = 250
VALID_SIZE: float = 0.2
PREPROCESS: list[Callable] = transforms.Compose(
    [
        transforms.functional.to_grayscale,
        transforms.PILToTensor(),
        transforms.Resize((36, 36), antialias=True),
        center_tensor,
        normalise,
    ]
)

In [ ]:
def handle_dataset(*filepaths: str | Path, **kwargs) -> tuple[DataLoader, DataLoader | None]:
    """Handles dataset(s). A dataset is loaded if its file exists, else is generated and saved."""
    raise NotImplementedError


try:
    train_ds, valid_ds = map(load_dataset, (f'{BASEPATH}/train_DS.pt', f'{BASEPATH}/valid_DS.pt'))
except FileNotFoundError:
    print('No dataset(s) found :c...\n')
    train_ds, valid_ds = get_dataset(
        f'{BASEPATH}/ImgsMockDatasetDMs', BATCH_SIZE, lambda x: ImageDataset(x, PREPROCESS), VALID_SIZE,
    )
    save_dataset(train_ds, save_to=f'{BASEPATH}/train_DS.pt')
    save_dataset(valid_ds, save_to=f'{BASEPATH}/valid_DS.pt')

Loading dataset...
Baking the dataset...


100%|██████████| 4000/4000 [00:00<00:00, 5744.38it/s]


Dataset ready-to-go!
Saving dataset...
Dataset saved!
Saving dataset...
Dataset saved!


In [ ]:
import torch.nn as nn

def get_conv_block(
    in_dims: int,
    out_dims: int,
    kernel_size: int,
    padding: int,
) -> nn.ModuleList:
    """Defines baseline conv block for mock model."""
    block = [
        nn.Conv2d(in_dims, out_dims, kernel_size, padding=padding),
        nn.BatchNorm2d(out_dims),
        nn.ReLU(),
    ]
    return nn.ModuleList(block)


class MockModel(nn.Module):
    """
    Baseline mock CNN model for `spark` API tests.
    """
    def __init__(
        self,
        in_dim: int,
        out_features: int,
        data_shape: tuple[int, int, int],
        maxpool: int = 2,
        dropout: float = 0.3, 
    ) -> None:
        super().__init__()
        in_features = int(data_shape.prod() / maxpool)
        self.net = nn.Sequential(
            *get_conv_block(in_dim, 16, 9, 4),
            *get_conv_block(16, 32, 7, 3),
            *get_conv_block(32, 32, 5, 2),
            *get_conv_block(32, in_dim, 5, 2),
            nn.MaxPool2d(maxpool),
            nn.Flatten(),
            nn.Linear(in_features, 1024), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(1024, out_features),
        )
    
    def forward(self, x: Tensor) -> Tensor:
        out = self.net(x)
        return out



def save_model(
    model: nn.Module,
    save_to: str | Path,
    info: dict[str, Any] | None = None,
    overwrite: bool = False,
) -> None:
    """Saves given model and its state to '.pt' file."""
    if Path(save_to).is_file() and not overwrite:
        print("Model already saved!")
        return None
    print("Saving model...")
    data = {
        'model': model,
        'model_state': model.state_dict(),
        **info,
    }
    torch.save(data, save_to)
    print("Model saved!")
    return None


def load_model(filepath: str | Path) -> dict[str, Any]:
    """Load given model and its state from '.pt' file."""
    print("Loading model...")
    model_state: dict = torch.load(filepath)
    print("Model loaded!")
    return model_state